In [1]:
# Monkey-patch to get live output from python file into this notebook
import subprocess
import sys

_original_run = subprocess.run

def notebook_run(args, **kwargs):
    process = subprocess.Popen(
        [str(arg) for arg in args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()

    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, args)

    return subprocess.CompletedProcess(args, return_code)

subprocess.run = notebook_run


In [2]:
# imports
from src.combine import run_colmap
import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
# run colmap
runit = run_colmap(image_dir="aloi_data/nerf_llff_data/fern/images", output_dir="outputs/demo")
runit.run()

I20260501 08:48:27.829969  1112 feature_extraction.cc:504] === Feature extraction ===
I20260501 08:48:27.832572 58140 sift.cc:757] Creating SIFT GPU feature extractor
I20260501 08:48:30.483708 49916 feature_extraction.cc:282] Processed file [1/20]
I20260501 08:48:30.483849 49916 feature_extraction.cc:285]   Name:            IMG_4026.JPG
I20260501 08:48:30.483868 49916 feature_extraction.cc:294]   Dimensions:      4032 x 3024
I20260501 08:48:30.483879 49916 feature_extraction.cc:297]   Camera:          #1 - SIMPLE_RADIAL
I20260501 08:48:30.483890 49916 feature_extraction.cc:300]   Focal Length:    3261.38px (Prior)
I20260501 08:48:30.483924 49916 feature_extraction.cc:304]   Features:        10936 (SIFT)
I20260501 08:48:30.718146 49916 feature_extraction.cc:318]   GPS:             LAT=37.872, LON=-122.262, ALT=150.016
I20260501 08:48:30.718303 49916 feature_extraction.cc:325]   Gravity:         X=0.000, Y=1.000, Z=0.000
I20260501 08:48:30.727506 49916 feature_extraction.cc:282] Processe

In [4]:
# visualize
ply_path = "outputs/demo/dense/fused.ply"

pcd = o3d.io.read_point_cloud(ply_path)

print(pcd)
print("Number of points:", len(pcd.points))

o3d.visualization.draw_geometries([pcd])

PointCloud with 1787638 points.
Number of points: 1787638


In [7]:
# remove noise
clean_pcd, noise_idx = runit.remove_noise()
# clean_pcd.paint_uniform_color([1, 0.706, 0])

# select noise
noise = pcd.select_by_index(noise_idx, invert=True)
noise.paint_uniform_color([1, 0, 0])

# visualize
o3d.visualization.draw_geometries([clean_pcd])